# Retrain YOLO11 for Face Recognition

This notebook fine-tunes YOLO11 on a custom dataset with two classes:

| Class ID | Label | Source |
|----------|-------|--------|
| 0 | `adnan` | Your selfie photos in `my_photos/` |
| 1 | `unknown_face` | Public faces from the [WIDER Face](http://shuoyang1213.me/WIDERFACE/) dataset |

**Workflow:**
1. Your selfie photos are pre-loaded in the `my_photos/` folder (no manual annotation needed).
2. We download a subset of the WIDER Face dataset for the "unknown_face" class.
3. A pre-trained YOLO11-face detector auto-annotates every image with bounding boxes.
4. We train YOLO11n on this two-class dataset. YOLO11's built-in augmentation (mosaic, mixup, flip, rotation) compensates for the small dataset size.
5. The trained model is exported to ONNX for cross-platform deployment.

In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

import os
import shutil
import random
import zipfile
from pathlib import Path
from ultralytics import YOLO
from huggingface_hub import hf_hub_download, snapshot_download

## Step 1: Verify your photos

Your selfie photos should already be in the `my_photos/` folder (uploaded by `deploy.sh` or `upload-to-workbench.sh`).

- Supported formats: `.jpg`, `.jpeg`, `.png`
- More variety (lighting, angles, backgrounds) improves training accuracy
- One face per photo works best

In [ ]:
MY_PHOTOS_DIR = Path("my_photos")
if not MY_PHOTOS_DIR.exists() or len(list(MY_PHOTOS_DIR.glob("*.jpg")) + list(MY_PHOTOS_DIR.glob("*.jpeg")) + list(MY_PHOTOS_DIR.glob("*.png"))) == 0:
    print("WARNING: No photos found in my_photos/ directory!")
    print("Please place ~50 selfie photos there before continuing.")
    print("Supported formats: .jpg, .jpeg, .png")
else:
    photo_count = len(list(MY_PHOTOS_DIR.glob("*.jpg")) + list(MY_PHOTOS_DIR.glob("*.jpeg")) + list(MY_PHOTOS_DIR.glob("*.png")))
    print(f"Found {photo_count} photos in my_photos/")

## Step 2: Load faces for the 'unknown_face' class

We combine two sources for the strongest possible unknown class:
1. **Real colleague photos** from `unknown_face/` — same events, lighting, and camera style as your selfies
2. **LFW portraits** from [Labeled Faces in the Wild](http://vis-www.cs.umass.edu/lfw/) — diverse close-up faces for broader generalization

In [ ]:
from sklearn.datasets import fetch_lfw_people
from PIL import Image
import numpy as np

UNKNOWN_DIR = Path("unknown_face")
LFW_DIR = Path("lfw_subset")
NUM_LFW = 200

# --- Source 1: Real colleague photos ---
custom_photos = []
if UNKNOWN_DIR.exists():
    custom_photos = sorted(UNKNOWN_DIR.glob("*.jpg")) + sorted(UNKNOWN_DIR.glob("*.jpeg")) + sorted(UNKNOWN_DIR.glob("*.png"))
print(f"Real colleague photos: {len(custom_photos)}")

# --- Source 2: LFW close-up portraits ---
print(f"Downloading LFW faces (close-up portraits)...")
lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True)
indices = list(range(len(lfw.images)))
random.shuffle(indices)
lfw_selected = indices[:NUM_LFW]
print(f"LFW portraits selected: {len(lfw_selected)}")

# --- Combine into lfw_subset/ ---
if LFW_DIR.exists():
    shutil.rmtree(LFW_DIR)
LFW_DIR.mkdir(exist_ok=True)

idx = 0
for p in custom_photos:
    shutil.copy2(p, LFW_DIR / f"unknown_{idx:04d}.jpg")
    idx += 1

for lfw_idx in lfw_selected:
    img_array = lfw.images[lfw_idx].astype(np.uint8)
    img = Image.fromarray(img_array)
    img.save(LFW_DIR / f"unknown_{idx:04d}.jpg", quality=95)
    idx += 1

print(f"\nCombined unknown_face class: {idx} photos")
print(f"  - {len(custom_photos)} real colleagues (close-match negatives)")
print(f"  - {len(lfw_selected)} LFW portraits (diverse negatives)")

## Step 3: Auto-annotate all images

Instead of manually drawing bounding boxes, we use a pre-trained [YOLO11-face](https://huggingface.co/AdamCodd/YOLOv11n-face-detection) model to detect faces in every image. For each detected face we write a YOLO-format label file (`class_id cx cy w h`) — then assign:

- **class 0** (`adnan`) to detections in your selfie photos
- **class 1** (`unknown_face`) to detections in the LFW photos

The dataset is split 80/20 into train and validation sets.

In [ ]:
DATASET_DIR = Path("face-dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True)

detector_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
detector = YOLO(detector_path)


def auto_annotate(image_path, class_id, output_img_dir, output_lbl_dir, prefix="img"):
    """Detect face, save image and YOLO-format label."""
    results = detector.predict(str(image_path), verbose=False, conf=0.3)
    if len(results[0].boxes) == 0:
        return False

    img_name = f"{prefix}_{image_path.stem}.jpg"
    lbl_name = f"{prefix}_{image_path.stem}.txt"

    shutil.copy2(image_path, output_img_dir / img_name)

    img_h, img_w = results[0].orig_shape
    with open(output_lbl_dir / lbl_name, "w") as f:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
    return True


user_photos = sorted(MY_PHOTOS_DIR.glob("*.jpg")) + sorted(MY_PHOTOS_DIR.glob("*.jpeg")) + sorted(MY_PHOTOS_DIR.glob("*.png"))
unknown_photos = sorted(LFW_DIR.glob("*.jpg"))

random.shuffle(user_photos)
random.shuffle(unknown_photos)

split_idx_user = int(len(user_photos) * 0.8)
split_idx_unknown = int(len(unknown_photos) * 0.8)

annotated = {"train": 0, "val": 0}
for photos, class_id, prefix in [
    (user_photos, 0, "adnan"),
    (unknown_photos, 1, "unknown"),
]:
    split_idx = int(len(photos) * 0.8)
    for split, photo_list in [("train", photos[:split_idx]), ("val", photos[split_idx:])]:
        for p in photo_list:
            if auto_annotate(p, class_id, DATASET_DIR / "images" / split, DATASET_DIR / "labels" / split, prefix):
                annotated[split] += 1

print(f"Dataset created: {annotated['train']} train, {annotated['val']} val images")

In [ ]:
data_yaml = DATASET_DIR / "data.yaml"
data_yaml.write_text(f"""path: {DATASET_DIR.resolve()}
train: images/train
val: images/val

nc: 2
names:
  0: adnan
  1: unknown_face
""")
print(f"Created {data_yaml}")
print(data_yaml.read_text())

## Step 4: Train the model

We fine-tune `yolo11m.pt` (the medium variant — 20.1M params, 7.7x more capacity than nano — fast, small) on our two-class dataset.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `epochs` | 15 | Enough for a small dataset with transfer learning |
| `imgsz` | 640 | Standard YOLO input resolution |
| `batch` | 4 | Conservative for CPU training |
| `device` | cpu | No GPU required |
| `mosaic` | 1.0 | Combines 4 images into one — great for small datasets |
| `mixup` | 0.3 | Blends two images to improve generalization |
| `fliplr` | 0.5 | Horizontal flip augmentation |
| `degrees` | 15.0 | Random rotation up to ±15° |
| `patience` | 5 | Early stopping if validation doesn't improve |

In [ ]:
model = YOLO("yolo11m.pt")

results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    mosaic=1.0,
    mixup=0.3,
    fliplr=0.5,
    degrees=15.0,
    patience=10,
    project="runs/face-recognition",
    name="train",
    exist_ok=True,
)

## Step 5: Export to ONNX

ONNX (Open Neural Network Exchange) is an open format that enables cross-platform inference. Exporting to ONNX lets us deploy the model on OpenVINO, TensorRT, or any ONNX-compatible runtime without depending on PyTorch at serving time.

In [ ]:
best_model_path = list(Path("runs").rglob("face-recognition/train/weights/best.pt"))
if best_model_path:
    best_pt = best_model_path[0]
    print(f"Found: {best_pt}")
    trained_model = YOLO(str(best_pt))
    onnx_path = trained_model.export(format="onnx")
    print(f"ONNX model exported to: {onnx_path}")
else:
    print("Training weights not found. Ensure training completed successfully.")

## Done!

The retrained model can now classify detected faces as **adnan** (class 0) or **unknown_face** (class 1).

**Outputs:**
- PyTorch weights: `runs/face-recognition/train/weights/best.pt`
- ONNX model: `runs/face-recognition/train/weights/best.onnx`

**Next:** Open **`03-test-retrained-model.ipynb`** to test the retrained model on new images and video.